# Compare Two-Module Communication Regimes

This notebook compares training results across three two-module configurations with different communication regimes:
- `two_module_rich_50_none`: No inter-module communication
- `two_module_rich_50_low`: Low communication (scale=0.05)
- `two_module_rich_50_high`: High communication (scale=0.1)

**Primary Focus**: Task switching effects when modules receive inputs (A1→B, B→A2)

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# Setup path
SCRIPT_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent  # transfer-interference directory
sys.path.insert(0, str(PROJECT_ROOT))

# Imports from project
from src.analysis import ann
from src.utils import figure_settings, figure_utils
from src.utils.figure_settings import schedule_colours, cm_conv, med_fontsize

# Setup figure output
figure_path = PROJECT_ROOT / "figures" / "compare_communication_regimes"
os.makedirs(figure_path, exist_ok=True)

# Set style
plt.style.use("default")
sns.set_palette("husl")
print("✓ Imports completed")

✓ Imports completed


## 1. Setup and Data Loading

Load data from all three communication regime conditions.

In [ ]:
# Define configuration names
configs = {
    'none': 'two_module_rich_50_none',
    'low': 'two_module_rich_50_low',
    'high': 'two_module_rich_50_high'
}

# Color scheme for conditions
comm_colors = {
    'none': '#808080',  # gray
    'low': '#56B4E9',   # blue
    'high': '#D9544D'   # red
}

# Load data for each condition
ann_data = {}
for comm_type, config_name in configs.items():
    data_path = PROJECT_ROOT / 'data' / 'simulations' / config_name
    if data_path.exists():
        ann_data[comm_type] = ann.load_ann_data(str(data_path))
        print(f"✓ Loaded {config_name}: {len(ann_data[comm_type]['same'])} participants")
    else:
        print(f"⚠ Warning: {data_path} does not exist")
        ann_data[comm_type] = {'same': [], 'near': [], 'far': []}

print(f"\nLoaded data for {len(ann_data)} communication regimes")

## 2. Task Switching Analysis

Analyze transfer costs and interference at task boundaries (A1→B, B→A2).

In [ ]:
def compute_retest_metrics(ann_data_dict):
    """Compute retest/interference metrics (A1 vs A2 performance)."""
    agg_data = []
    
    for comm_type, data in ann_data_dict.items():
        for schedule_name, schedule_data in data.items():
            for subj in range(len(schedule_data)):
                subj_data = schedule_data[subj]
                
                # Get accuracy by task section (winter responses only, feature_idx==1)
                A1_accuracy = subj_data['accuracy'][0, 1::2].copy()  # Phase 0, every other (winter)
                A2_accuracy = subj_data['accuracy'][2, 1::2].copy()  # Phase 2, every other (winter)
                
                # Average over relevant windows
                final_A1_acc = np.mean(A1_accuracy[-6:])  # final 6 trials of A1
                initial_A2_acc = np.mean(A2_accuracy[0:6])  # first 6 trials of A2
                mean_A2_acc = np.mean(A2_accuracy)  # overall A2 performance
                
                # Calculate retest/interference metrics
                retest_error_diff = initial_A2_acc - final_A1_acc  # Initial A2 vs final A1
                interference = final_A1_acc - mean_A2_acc  # Final A1 vs mean A2
                
                agg_data.append({
                    'participant': str(subj_data['participant']),
                    'condition': schedule_name,
                    'comm_type': comm_type,
                    'final_A1_acc': final_A1_acc,
                    'initial_A2_acc': initial_A2_acc,
                    'mean_A2_acc': mean_A2_acc,
                    'retest_error_diff': retest_error_diff,
                    'interference': interference
                })
    
    return pd.DataFrame(agg_data)

# Compute transfer and retest metrics
transfer_dfs = {}
retest_dfs = {}

for comm_type, data in ann_data.items():
    if len(data['same']) > 0:  # Only if data exists
        transfer_dfs[comm_type] = ann.compute_transfer_anns(data)
        transfer_dfs[comm_type]['comm_type'] = comm_type

# Combine transfer data
if transfer_dfs:
    transfer_combined = pd.concat(transfer_dfs.values(), ignore_index=True)
    transfer_combined.rename(columns={'error_diff': 'transfer_error_diff'}, inplace=True)
    
    # Compute retest metrics
    retest_combined = compute_retest_metrics(ann_data)
    
    print("Transfer metrics summary:")
    print(transfer_combined.groupby('comm_type')['transfer_error_diff'].describe())
    print("\nRetest metrics summary:")
    print(retest_combined.groupby('comm_type')['retest_error_diff'].describe())
else:
    print("No data available for analysis")
if 'transfer_combined' in locals() and len(transfer_combined) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(10*cm_conv, 4*cm_conv))
    
    # Transfer error difference (A1→B)
    ax1 = axes[0]
    comm_order = ['none', 'low', 'high']
    sns.stripplot(data=transfer_combined, x='comm_type', y='transfer_error_diff', 
                  ax=ax1, order=comm_order, palette=[comm_colors[c] for c in comm_order],
                  alpha=0.5, size=3, jitter=True, zorder=1)
    sns.pointplot(data=transfer_combined, x='comm_type', y='transfer_error_diff',
                  ax=ax1, order=comm_order, color='black', markersize=4, zorder=2)
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax1.set_xlabel('Communication Regime')
    ax1.set_ylabel('Transfer Error Difference\n(A1→B)')
    ax1.set_title('Transfer Cost at A1→B Switch')
    figure_utils._style_axes(ax1)
    
    # Retest error difference (A1→A2)
    ax2 = axes[1]
    retest_same = retest_combined[retest_combined['condition'] == 'same']
    sns.stripplot(data=retest_same, x='comm_type', y='retest_error_diff',
                  ax=ax2, order=comm_order, palette=[comm_colors[c] for c in comm_order],
                  alpha=0.5, size=3, jitter=True, zorder=1)
    sns.pointplot(data=retest_same, x='comm_type', y='retest_error_diff',
                  ax=ax2, order=comm_order, color='black', markersize=4, zorder=2)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax2.set_xlabel('Communication Regime')
    ax2.set_ylabel('Retest Error Difference\n(A1→A2)')
    ax2.set_title('Interference at B→A2 Switch')
    figure_utils._style_axes(ax2)
    
    plt.tight_layout()
    plt.savefig(figure_path / 'task_switching_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No data available for visualization")

In [ ]:
# Visualize transfer and retest metrics

In [ ]:
def extract_learning_curves(ann_data_dict, phase_idx=0, metric='accuracy', feature_idx=1):
    """Extract learning curves for a specific phase."""
    curves = {}
    
    for comm_type, data in ann_data_dict.items():
        curves[comm_type] = []
        for schedule_name, schedule_data in data.items():
            for subj in range(len(schedule_data)):
                subj_data = schedule_data[subj]
                if metric == 'accuracy':
                    # Extract accuracy for specific feature (winter: feature_idx=1)
                    curve = subj_data['accuracy'][phase_idx, feature_idx::2].copy()
                elif metric == 'loss':
                    curve = subj_data['losses'][phase_idx, :].copy()
                else:
                    raise ValueError(f"Unknown metric: {metric}")
                curves[comm_type].append(curve)
    
    return curves

# Extract learning curves for each phase
phases = {'A1': 0, 'B': 1, 'A2': 2}
learning_curves = {}

for phase_name, phase_idx in phases.items():
    learning_curves[phase_name] = {
        'accuracy': extract_learning_curves(ann_data, phase_idx, 'accuracy'),
        'loss': extract_learning_curves(ann_data, phase_idx, 'loss')
    }

print("Learning curves extracted for all phases")

## 4. Module-Specific Activation Analysis

Analyze how Module A and Module B activations change across phases and communication regimes.

## 4. Module-Specific Activation Analysis

Analyze how Module A and Module B activations change across phases and communication regimes.

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(2, 3, figsize=(15*cm_conv, 8*cm_conv))

for phase_idx, (phase_name, phase_num) in enumerate(phases.items()):
    # Accuracy curves
    ax_acc = axes[0, phase_idx]
    for comm_type in ['none', 'low', 'high']:
        if comm_type in learning_curves[phase_name]['accuracy']:
            curves = learning_curves[phase_name]['accuracy'][comm_type]
            if len(curves) > 0:
                # Find minimum length
                min_len = min(len(c) for c in curves)
                curves_aligned = [c[:min_len] for c in curves]
                curves_array = np.array(curves_aligned)
                
                # Plot mean and std
                mean_curve = np.mean(curves_array, axis=0)
                std_curve = np.std(curves_array, axis=0)
                x = np.arange(len(mean_curve))
                
                ax_acc.plot(x, mean_curve, color=comm_colors[comm_type], 
                           label=comm_type, linewidth=1.5)
                ax_acc.fill_between(x, mean_curve - std_curve, mean_curve + std_curve,
                                   color=comm_colors[comm_type], alpha=0.2)
    
    ax_acc.set_xlabel('Trial')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title(f'{phase_name} Phase - Accuracy')
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)
    figure_utils._style_axes(ax_acc)
    
    # Loss curves
    ax_loss = axes[1, phase_idx]
    for comm_type in ['none', 'low', 'high']:
        if comm_type in learning_curves[phase_name]['loss']:
            curves = learning_curves[phase_name]['loss'][comm_type]
            if len(curves) > 0:
                min_len = min(len(c) for c in curves)
                curves_aligned = [c[:min_len] for c in curves]
                curves_array = np.array(curves_aligned)
                
                mean_curve = np.mean(curves_array, axis=0)
                std_curve = np.std(curves_array, axis=0)
                x = np.arange(len(mean_curve))
                
                ax_loss.plot(x, mean_curve, color=comm_colors[comm_type],
                           label=comm_type, linewidth=1.5)
                ax_loss.fill_between(x, mean_curve - std_curve, mean_curve + std_curve,
                                   color=comm_colors[comm_type], alpha=0.2)
    
    ax_loss.set_xlabel('Trial')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_title(f'{phase_name} Phase - Loss')
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)
    figure_utils._style_axes(ax_loss)

plt.tight_layout()
plt.savefig(figure_path / 'learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize module activations
if activation_stats and any(len(activation_stats[ct]['A1']['mod_A']) > 0 for ct in activation_stats):
    fig, axes = plt.subplots(2, 3, figsize=(15*cm_conv, 8*cm_conv))
    
    phases_list = ['A1', 'B', 'A2']
    
    for phase_idx, phase_name in enumerate(phases_list):
        # Module A activations
        ax_A = axes[0, phase_idx]
        for comm_type in ['none', 'low', 'high']:
            if comm_type in activation_stats and len(activation_stats[comm_type][phase_name]['mod_A']) > 0:
                all_activations = np.concatenate(activation_stats[comm_type][phase_name]['mod_A'])
                mean_act = np.mean(all_activations)
                std_act = np.std(all_activations)
                
                ax_A.bar(comm_type, mean_act, color=comm_colors[comm_type], alpha=0.7,
                        yerr=std_act, capsize=5, label=comm_type)
        
        ax_A.set_ylabel('Activation Magnitude (L2 norm)')
        ax_A.set_title(f'Module A - {phase_name}')
        ax_A.set_xticklabels(['none', 'low', 'high'])
        figure_utils._style_axes(ax_A)
        
        # Module B activations
        ax_B = axes[1, phase_idx]
        for comm_type in ['none', 'low', 'high']:
            if comm_type in activation_stats and len(activation_stats[comm_type][phase_name]['mod_B']) > 0:
                all_activations = np.concatenate(activation_stats[comm_type][phase_name]['mod_B'])
                mean_act = np.mean(all_activations)
                std_act = np.std(all_activations)
                
                ax_B.bar(comm_type, mean_act, color=comm_colors[comm_type], alpha=0.7,
                        yerr=std_act, capsize=5, label=comm_type)
        
        ax_B.set_ylabel('Activation Magnitude (L2 norm)')
        ax_B.set_title(f'Module B - {phase_name}')
        ax_B.set_xticklabels(['none', 'low', 'high'])
        figure_utils._style_axes(ax_B)
    
    plt.tight_layout()
    plt.savefig(figure_path / 'module_activations.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No module activation data available")

## 5. Representational Geometry Analysis

Compare principal angles and PCA visualizations across communication regimes.

In [ ]:
# Analyze cross-module activation (inactive module receiving communication)
def analyze_cross_module_activation(ann_data_dict):
    """Analyze how much inactive modules activate via communication."""
    cross_activation = {}
    
    for comm_type, data in ann_data_dict.items():
        cross_activation[comm_type] = {
            'A_active_B_receives': [],  # During A1/A2, how much does B activate?
            'B_active_A_receives': []   # During B, how much does A activate?
        }
        
        for schedule_name, schedule_data in data.items():
            for subj in range(len(schedule_data)):
                subj_data = schedule_data[subj]
                
                if 'hiddens_A' in subj_data and 'hiddens_B' in subj_data:
                    h_A = subj_data['hiddens_A']
                    h_B = subj_data['hiddens_B']
                    
                    # During A1/A2 (Module A active), measure Module B activation
                    for phase_idx in [0, 2]:  # A1 and A2
                        if h_B.shape[0] > phase_idx:
                            mod_B_acts = np.linalg.norm(h_B[phase_idx, :, :], axis=1)
                            cross_activation[comm_type]['A_active_B_receives'].extend(mod_B_acts)
                    
                    # During B (Module B active), measure Module A activation
                    if h_A.shape[0] > 1:
                        mod_A_acts = np.linalg.norm(h_A[1, :, :], axis=1)
                        cross_activation[comm_type]['B_active_A_receives'].extend(mod_A_acts)
    
    return cross_activation

cross_activation = analyze_cross_module_activation(ann_data)

# Visualize cross-module activation
if cross_activation and any(len(cross_activation[ct]['A_active_B_receives']) > 0 for ct in cross_activation):
    fig, axes = plt.subplots(1, 2, figsize=(10*cm_conv, 4*cm_conv))
    
    # Module B activation when A is active
    ax1 = axes[0]
    for comm_type in ['none', 'low', 'high']:
        if comm_type in cross_activation and len(cross_activation[comm_type]['A_active_B_receives']) > 0:
            acts = cross_activation[comm_type]['A_active_B_receives']
            mean_act = np.mean(acts)
            std_act = np.std(acts)
            ax1.bar(comm_type, mean_act, color=comm_colors[comm_type], alpha=0.7,
                   yerr=std_act, capsize=5)
    ax1.set_ylabel('Module B Activation\n(when A is active)')
    ax1.set_title('Cross-Module Communication: A→B')
    ax1.set_xticklabels(['none', 'low', 'high'])
    figure_utils._style_axes(ax1)
    
    # Module A activation when B is active
    ax2 = axes[1]
    for comm_type in ['none', 'low', 'high']:
        if comm_type in cross_activation and len(cross_activation[comm_type]['B_active_A_receives']) > 0:
            acts = cross_activation[comm_type]['B_active_A_receives']
            mean_act = np.mean(acts)
            std_act = np.std(acts)
            ax2.bar(comm_type, mean_act, color=comm_colors[comm_type], alpha=0.7,
                   yerr=std_act, capsize=5)
    ax2.set_ylabel('Module A Activation\n(when B is active)')
    ax2.set_title('Cross-Module Communication: B→A')
    ax2.set_xticklabels(['none', 'low', 'high'])
    figure_utils._style_axes(ax2)
    
    plt.tight_layout()
    plt.savefig(figure_path / 'cross_module_activation.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No cross-module activation data available")

In [ ]:
# Compute principal angles for each condition
principal_angles = {}

for comm_type, data in ann_data.items():
    if len(data['same']) > 0:
        pa_df = ann.get_principal_angles(data)
        pa_df['comm_type'] = comm_type
        principal_angles[comm_type] = pa_df

# Combine principal angle data
if principal_angles:
    pa_combined = pd.concat(principal_angles.values(), ignore_index=True)
    
    print("Principal angles summary:")
    print(pa_combined.groupby('comm_type')[['within_task_A', 'within_task_B', 'across_task']].describe())
    
    # Visualize principal angles
    fig, axes = plt.subplots(1, 3, figsize=(15*cm_conv, 4*cm_conv))
    
    metrics = ['within_task_A', 'within_task_B', 'across_task']
    metric_labels = ['Within Task A', 'Within Task B', 'Across Task']
    
    for ax, metric, label in zip(axes, metrics, metric_labels):
        pa_same = pa_combined[pa_combined['condition'] == 'same']
        sns.stripplot(data=pa_same, x='comm_type', y=metric,
                     ax=ax, order=['none', 'low', 'high'],
                     palette=[comm_colors[c] for c in ['none', 'low', 'high']],
                     alpha=0.5, size=3, jitter=True, zorder=1)
        sns.pointplot(data=pa_same, x='comm_type', y=metric,
                     ax=ax, order=['none', 'low', 'high'],
                     color='black', markersize=4, zorder=2)
        ax.set_xlabel('Communication Regime')
        ax.set_ylabel('Principal Angle (radians)')
        ax.set_title(label)
        figure_utils._style_axes(ax)
    
    plt.tight_layout()
    plt.savefig(figure_path / 'principal_angles.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No principal angle data available")

## 6. Statistical Comparisons

Perform statistical tests comparing communication regimes.

In [ ]:
def compute_effect_size(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    d = (mean1 - mean2) / pooled_std
    return d

# Statistical comparisons
if 'transfer_combined' in locals() and len(transfer_combined) > 0:
    print("=" * 60)
    print("STATISTICAL COMPARISONS")
    print("=" * 60)
    
    # Transfer cost comparisons
    print("\n1. Transfer Cost (A1→B) Comparisons:")
    print("-" * 60)
    
    transfer_same = transfer_combined[transfer_combined['condition'] == 'same']
    
    for comparison in [('none', 'low'), ('none', 'high'), ('low', 'high')]:
        g1 = transfer_same[transfer_same['comm_type'] == comparison[0]]['transfer_error_diff'].values
        g2 = transfer_same[transfer_same['comm_type'] == comparison[1]]['transfer_error_diff'].values
        
        if len(g1) > 0 and len(g2) > 0:
            t_stat, p_val = stats.ttest_ind(g1, g2)
            effect_size = compute_effect_size(g1, g2)
            
            print(f"{comparison[0]} vs {comparison[1]}:")
            print(f"  t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")
            print(f"  Effect size (Cohen's d): {effect_size:.3f}")
            print(f"  Mean difference: {np.mean(g1) - np.mean(g2):.4f}")
            print()
    
    # Retest/interference comparisons
    if 'retest_combined' in locals() and len(retest_combined) > 0:
        print("\n2. Retest/Interference (A1→A2) Comparisons:")
        print("-" * 60)
        
        retest_same = retest_combined[retest_combined['condition'] == 'same']
        
        for comparison in [('none', 'low'), ('none', 'high'), ('low', 'high')]:
            g1 = retest_same[retest_same['comm_type'] == comparison[0]]['retest_error_diff'].values
            g2 = retest_same[retest_same['comm_type'] == comparison[1]]['retest_error_diff'].values
            
            if len(g1) > 0 and len(g2) > 0:
                t_stat, p_val = stats.ttest_ind(g1, g2)
                effect_size = compute_effect_size(g1, g2)
                
                print(f"{comparison[0]} vs {comparison[1]}:")
                print(f"  t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")
                print(f"  Effect size (Cohen's d): {effect_size:.3f}")
                print(f"  Mean difference: {np.mean(g1) - np.mean(g2):.4f}")
                print()
    
    # Principal angle comparisons
    if 'pa_combined' in locals() and len(pa_combined) > 0:
        print("\n3. Principal Angle Comparisons (Across-Task):")
        print("-" * 60)
        
        pa_same = pa_combined[pa_combined['condition'] == 'same']
        
        for comparison in [('none', 'low'), ('none', 'high'), ('low', 'high')]:
            g1 = pa_same[pa_same['comm_type'] == comparison[0]]['across_task'].values
            g2 = pa_same[pa_same['comm_type'] == comparison[1]]['across_task'].values
            
            if len(g1) > 0 and len(g2) > 0:
                t_stat, p_val = stats.ttest_ind(g1, g2)
                effect_size = compute_effect_size(g1, g2)
                
                print(f"{comparison[0]} vs {comparison[1]}:")
                print(f"  t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")
                print(f"  Effect size (Cohen's d): {effect_size:.3f}")
                print(f"  Mean difference: {np.mean(g1) - np.mean(g2):.4f}")
                print()
else:
    print("No data available for statistical comparisons")

In [ ]:
# Create summary table
if 'transfer_combined' in locals() and 'retest_combined' in locals():
    summary_data = []
    
    for comm_type in ['none', 'low', 'high']:
        # Transfer metrics
        transfer_subset = transfer_combined[
            (transfer_combined['comm_type'] == comm_type) & 
            (transfer_combined['condition'] == 'same')
        ]
        
        # Retest metrics
        retest_subset = retest_combined[
            (retest_combined['comm_type'] == comm_type) &
            (retest_combined['condition'] == 'same')
        ]
        
        if len(transfer_subset) > 0 and len(retest_subset) > 0:
            summary_data.append({
                'Communication Regime': comm_type,
                'Transfer Cost (A1→B)': f"{np.mean(transfer_subset['transfer_error_diff']):.4f} ± {np.std(transfer_subset['transfer_error_diff']):.4f}",
                'Interference (A1→A2)': f"{np.mean(retest_subset['retest_error_diff']):.4f} ± {np.std(retest_subset['retest_error_diff']):.4f}",
                'Final A1 Accuracy': f"{np.mean(retest_subset['final_A1_acc']):.4f} ± {np.std(retest_subset['final_A1_acc']):.4f}",
                'Mean A2 Accuracy': f"{np.mean(retest_subset['mean_A2_acc']):.4f} ± {np.std(retest_subset['mean_A2_acc']):.4f}",
            })
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        print("=" * 80)
        print("SUMMARY TABLE")
        print("=" * 80)
        print(summary_df.to_string(index=False))
        print()
        
        # Key insights
        print("=" * 80)
        print("KEY INSIGHTS")
        print("=" * 80)
        print("\n1. Transfer Cost (A1→B):")
        print("   - Measures performance drop when switching from Task A to Task B")
        print("   - Lower (more negative) values indicate better transfer")
        print("   - Communication may help Module B leverage Module A's knowledge")
        
        print("\n2. Interference (A1→A2):")
        print("   - Measures performance change when returning to Task A after B training")
        print("   - Negative values indicate catastrophic forgetting")
        print("   - Communication may help preserve Module A representations during B training")
        
        print("\n3. Communication Effects:")
        print("   - None: Modules are completely isolated")
        print("   - Low: Minimal information flow (scale=0.05)")
        print("   - High: Strong information flow (scale=0.1)")
        
        print("\n4. Questions to Consider:")
        print("   - Does communication reduce transfer cost?")
        print("   - Does communication prevent catastrophic forgetting?")
        print("   - Are there trade-offs between transfer and interference?")
        print("   - How does communication strength affect these metrics?")
    else:
        print("No summary data available")
else:
    print("No data available for summary")

## 7. Summary and Insights

Summary table of key metrics and interpretation of communication effects.